# 1. Set the env/ load libraries/packages

In [ ]:
import os
import warnings
from pathlib import Path

import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import chi2_contingency

from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import cross_val_score


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# file_path1 = '/content/drive/MyDrive/C03_project/bigdata/bigdata_최종프로젝트/data/bat_process.csv'
# file_path2 = '/content/drive/MyDrive/C03_project/bigdata/bigdata_최종프로젝트/data/bat_tat.csv'

file_path1 = './data/bat_process.csv'
file_path2 = './data/bat_tat.csv'

df_process = pd.read_csv(file_path1, encoding='euc-kr')
df_bat = pd.read_csv(file_path2, encoding='euc-kr')

Mounted at /content/drive


In [3]:
df_process.head()

,lot_id,tray_id,cell_id,dt_start,judge,rta1_cell_no,rta1_box_col,rta1_box_row,rta1_box_dan,hta1_box_col,...,sa3_box_row,sa3_box_dan,socv3_ocv,ocv2_deltaocv,ocv1_deltaocv,m1_thick,m1_voltage,m1_res_ac,m1_mv,m1_voltage_an
0,LOT-100,TRAY-1023,CELL-10747,01SEP25:10:01:25,양품,10,08열,33연,08단,03열,...,37연,04단,3759.3,5.0,6.8,4718.0,37583.0,94.0,565.0,164.0
1,LOT-100,TRAY-1023,CELL-10748,01SEP25:10:01:25,양품,7,08열,33연,08단,03열,...,37연,04단,3758.4,5.0,6.8,4718.0,37574.0,94.0,565.0,73.0
2,LOT-100,TRAY-1023,CELL-10749,01SEP25:10:01:25,양품,19,08열,33연,08단,03열,...,37연,04단,3759.0,5.1,6.9,4700.0,37579.0,91.0,585.0,-57.0
3,LOT-100,TRAY-1023,CELL-10750,01SEP25:10:01:25,양품,24,08열,33연,08단,03열,...,37연,04단,3759.5,5.1,6.9,4688.0,37583.0,93.0,606.0,99.0
4,LOT-100,TRAY-1023,CELL-10751,01SEP25:10:01:25,양품,17,08열,33연,08단,03열,...,37연,04단,3758.2,5.0,6.8,4696.0,37570.0,91.0,605.0,36.0


In [4]:
df_bat.head()

,lot_id,tray_id,rta1_tat,hta1_tat,rta2_tat,ocv1_tat,c1_tat,dc1_tat,c2_tat,dc2_tat,...,c4_tat,ocv2_tat,pg1_tat,pc1_tat,sa1_tat,socv1_tat,sa2_tat,socv2_tat,sa3_tat,socv3_tat
0,LOT-100,TRAY-1002,217709,86476,4524,118,3580,405,453,405,...,3489,118,291,291,53555,5,355111,5,215651,6
1,LOT-100,TRAY-1004,217663,86444,4916,118,3555,405,456,405,...,3441,118,290,290,54095,4,355249,5,215782,5
2,LOT-100,TRAY-1005,217504,86456,4872,119,3553,405,454,404,...,3435,118,290,291,54316,5,354928,6,215655,5
3,LOT-100,TRAY-1007,217901,86481,4978,119,3558,405,459,405,...,3409,118,290,290,54522,5,355271,5,215550,5
4,LOT-100,TRAY-1008,217901,86480,4978,118,3556,405,465,405,...,3411,118,290,290,54522,5,355271,5,215550,5


In [5]:
print("df_process columns:")
print(df_process.columns.tolist())

print("\ndf_bat columns:")
print(df_bat.columns.tolist())

# 2. 공통 컬럼 확인
common_cols = set(df_process.columns) & set(df_bat.columns)
common_cols = list(common_cols)

print("\n공통 컬럼:")
print(common_cols)

df_process columns:
['lot_id', 'tray_id', 'cell_id', 'dt_start', 'judge', 'rta1_cell_no', 'rta1_box_col', 'rta1_box_row', 'rta1_box_dan', 'hta1_box_col', 'hta1_box_row', 'hta1_box_dan', 'rta2_box_col', 'rta2_box_row', 'rta2_box_dan', 'ocv1_ocv', 'ocv1_box_col', 'ocv1_box_dan', 'c1_curr_end', 'c1_voltage_avg', 'c1_capa', 'c1_ccval', 'c1_time_cc', 'c1_box_col', 'c1_box_dan', 'c1_temp_avg', 'dc1_curr_end', 'dc1_voltage_avg', 'dc1_capa', 'dc1_box_col', 'dc1_box_dan', 'dc1_temp_avg', 'dc1_capafit', 'c2_curr_end', 'c2_voltage_avg', 'c2_capa', 'c2_ccval', 'c2_time_cc', 'c2_box_col', 'c2_box_dan', 'c2_temp_avg', 'dc2_curr_end', 'dc2_voltage_avg', 'dc2_capa', 'dc2_box_col', 'dc2_box_dan', 'dc2_temp_avg', 'dc2_capafit', 'c3_curr_end', 'c3_voltage_avg', 'c3_capa', 'c3_ccval', 'c3_time_cv', 'c3_cvval', 'c3_time_cc', 'c3_box_col', 'c3_box_dan', 'c3_temp_avg', 'dc3_curr_end', 'dc3_voltage_avg', 'dc3_capa', 'dc3_box_col', 'dc3_box_dan', 'dc3_temp_avg', 'dc3_capafit', 'c4_curr_end', 'c4_voltage_avg', 

In [6]:
# 3. 공통 컬럼별 고유값 개수, 중복 여부 확인
for col in common_cols:
    print(f"\n[{col}]")
    print("df_process unique:", df_process[col].nunique())
    print("df_process rows:", len(df_process))
    print("df_process duplicated:", df_process[col].duplicated().sum())

    print("df_bat unique:", df_bat[col].nunique())
    print("df_bat rows:", len(df_bat))
    print("df_bat duplicated:", df_bat[col].duplicated().sum())


[tray_id]
df_process unique: 1780
df_process rows: 38595
df_process duplicated: 36815
df_bat unique: 1780
df_bat rows: 1797
df_bat duplicated: 17

[lot_id]
df_process unique: 45
df_process rows: 38595
df_process duplicated: 38550
df_bat unique: 45
df_bat rows: 1797
df_bat duplicated: 1752


In [7]:
keys = ['lot_id', 'tray_id']

print("df_process 행 수:", len(df_process))
print("df_bat 행 수:", len(df_bat))

print("\ndf_process key 조합 unique:")
print(df_process[keys].drop_duplicates().shape[0])

print("\ndf_bat key 조합 unique:")
print(df_bat[keys].drop_duplicates().shape[0])

print("\ndf_process key 조합 중복 수:")
print(df_process.duplicated(subset=keys).sum())

print("\ndf_bat key 조합 중복 수:")
print(df_bat.duplicated(subset=keys).sum())

df_process 행 수: 38595
df_bat 행 수: 1797

df_process key 조합 unique:
1797

df_bat key 조합 unique:
1797

df_process key 조합 중복 수:
36798

df_bat key 조합 중복 수:
0


In [8]:
keys = ['lot_id', 'tray_id']

df_final = pd.merge(
    df_process,
    df_bat,
    on=keys,
    how='left',
    validate='many_to_one'
)

print("df_process 행 수:", len(df_process))
print("df_final 행 수:", len(df_final))
print("증가 행 수:", len(df_final) - len(df_process))

df_final.head()

df_process 행 수: 38595
df_final 행 수: 38595
증가 행 수: 0


,lot_id,tray_id,cell_id,dt_start,judge,rta1_cell_no,rta1_box_col,rta1_box_row,rta1_box_dan,hta1_box_col,...,c4_tat,ocv2_tat,pg1_tat,pc1_tat,sa1_tat,socv1_tat,sa2_tat,socv2_tat,sa3_tat,socv3_tat
0,LOT-100,TRAY-1023,CELL-10747,01SEP25:10:01:25,양품,10,08열,33연,08단,03열,...,3428,118,290,290,53500,5,355357,5,215150,5
1,LOT-100,TRAY-1023,CELL-10748,01SEP25:10:01:25,양품,7,08열,33연,08단,03열,...,3428,118,290,290,53500,5,355357,5,215150,5
2,LOT-100,TRAY-1023,CELL-10749,01SEP25:10:01:25,양품,19,08열,33연,08단,03열,...,3428,118,290,290,53500,5,355357,5,215150,5
3,LOT-100,TRAY-1023,CELL-10750,01SEP25:10:01:25,양품,24,08열,33연,08단,03열,...,3428,118,290,290,53500,5,355357,5,215150,5
4,LOT-100,TRAY-1023,CELL-10751,01SEP25:10:01:25,양품,17,08열,33연,08단,03열,...,3428,118,290,290,53500,5,355357,5,215150,5


2. Data Preprocessing

In [9]:
# 원본 연속형 변수만 (파생변수 제외)
raw_num_cols = [
    socv1_tat, sa2_tat, socv2_tat, sa3_tat, socv3_tat, sa4_tat, socv4_tat, sa5_tat
]

corr_raw = df_final[raw_num_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_raw,
    cmap='coolwarm',
    vmin=-1, vmax=1,
    annot=True, fmt='.2f',
    linewidths=0.5,
    square=True,
    cbar_kws={'label': 'correlation'}
)
plt.title('Original Numeric Variables Correlation')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

NameError: name 'socv1_tat' is not defined